In [82]:
%pip install pandas transformers deepparse huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [83]:
import torch
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
    device = torch.device('cuda')
elif torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+xpu, Device: xpu


In [84]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import pandas as pd
from collections import OrderedDict

COLS_TO_PREDICT = [
    "HouseNumber",
    "StreetName",
    "City",
    "State",
    "Country"
]

EXCEPT_COUNTRY = COLS_TO_PREDICT[:-1]

# Mahsa's code for levenshtein distance
def levenshtein(a: str, b: str, case_insensitive=True) -> int:
    # If one of the strings is empty
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    
    if case_insensitive:
        a = a.lower()
        b = b.lower()

    # Create distance matrix (size: (len(a)+1) x (len(b)+1))
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    # Initialize first row/column
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    # Fill in matrix
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )
    distance = dp[-1][-1]

    return distance

def compare_preds(preds : pd.DataFrame, labels : pd.DataFrame, target_columns = COLS_TO_PREDICT, ignore_trash_columns = True):
    # Drop meta columns that may be included in the preds dataframe
    labels = labels.astype(str)

    tolerance_levels = 5
    correct_with_tol = [0,] * tolerance_levels
    total_rows = 0
    prediction_count = 0
    label_count = 0
    true_positives = 0
    
    sum_levenshtein = 0
    sum_similarity = 0.0
    sum_levenshtein_match = 0
    sum_similarity_match = 0.0
    some_match_count = 0
    
    if not ignore_trash_columns:
        # labels that should not have been predicted at all
        trash_predictions = preds[[col for col in preds.columns if col not in labels.columns]].stack()
        trash_count = trash_predictions.notna().sum()
        total_rows += trash_count
        prediction_count += trash_count
        sum_levenshtein += trash_predictions.dropna().astype(str).str.len().sum()
    for col in target_columns:
        total_rows += len(labels)
        label_count += labels[col].notna().sum()
        if col not in preds.columns:
            # all missing predictions are incorrect
            sum_levenshtein += labels[col].dropna().str.len().sum()
        else:
            prediction_count += preds[col].notna().sum()
            strings_to_compare = pd.concat([preds[col].fillna(""), labels[col].fillna("")], axis=1)
            levenshtein_scores = strings_to_compare.apply(
                lambda row: levenshtein(row.iloc[0], row.iloc[1]), axis=1
            )
            levenshtein_bounds = strings_to_compare.apply(
                lambda row: max(len(row.iloc[0]), len(row.iloc[1])), axis=1
            )
            similarity = ((levenshtein_bounds - levenshtein_scores) / levenshtein_bounds).fillna(1.0) # nan => div by 0 => both are empty strings => similarity 1.0
            sum_levenshtein += levenshtein_scores.sum()
            sum_similarity += similarity.sum()
            sum_levenshtein_match += levenshtein_scores[similarity >= 0].sum()
            sum_similarity_match += similarity[similarity >= 0].sum()
            some_match_count += (similarity > 0).sum()
            true_positives += ((levenshtein_scores == 0) & preds[col].notna() & labels[col].notna()).sum()
            for tol in range(tolerance_levels):
                correct_with_tol[tol] += (levenshtein_scores <= tol).sum()
    results = OrderedDict()
    results["accuracy"] = correct_with_tol[0] / total_rows
    results["precision"] = true_positives / prediction_count if prediction_count > 0 else 0.0
    results["recall"] = true_positives / label_count if label_count > 0 else 0.0
    results["f1"] = 2 * results["precision"] * results["recall"] / (results["precision"] + results["recall"]) if (results["precision"] + results["recall"]) > 0 else 0.0
    for tol in range(1, tolerance_levels):
        results[f"accuracy_with_tol_{tol}"] = correct_with_tol[tol] / total_rows
    results["average_levenshtein"] = sum_levenshtein / total_rows
    results["average_similarity"] = sum_similarity / total_rows
    results["average_levenshtein_match"] = sum_levenshtein_match / some_match_count if some_match_count > 0 else 0.0
    results["average_similarity_match"] = sum_similarity_match / some_match_count if some_match_count > 0 else 0.0
    results["no_match_rate"] = 1.0 - (some_match_count / total_rows)
    return results

In [86]:
class ParsedAddressResultBuilder:
    def __init__(self):
        self.inner = {}
    
    def add_part(self, label, part):
        if not part: # ignore None or empty
            return
        part = str(part)
        if label in ["fullResponse", "model-fullAddress", "error"]:
            print(f"ERROR: Attempt to add reserved label: {label}")
            return
        if label == "fullAddress":
            label = "model-fullAddress" # avoid collision but keep the wrongfully generated field
        conflict = self.inner.get(label, None)
        if conflict:
            if conflict != part: # only add if different
                self.inner[label] += "___" + part
        else:
            self.inner[label] = part

    def set_reserved(self, label, value):
        self.inner[label] = value

    def build(self) -> dict:
        return self.inner

    


In [87]:
import deepparse.parser

addr_parser = deepparse.parser.AddressParser(device=torch.get_default_device())

print(addr_parser("123 Main Street, Los Angeles, CA, USA").to_dict())
del addr_parser # release gpu resources

c:\Users\rpa\Documents\wiedergutmachung\bzk-post-processing\.conda\Lib\site-packages\deepparse\download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
{'StreetNumber': '123', 'StreetName': 'main street los angeles', 'Unit': None, 'Municipality': 'ca', 'Province': 'usa', 'PostalCode': None, 'Orientation': None, 'GeneralDelivery': None, 'EOS': None}


In [88]:

DEEPPARSE_LABEL_MAPPING = {
    "StreetNumber": "HouseNumber",
    # "StreetName": "StreetName",
    "Municipality": "City",
    #"Province": "State", # note: DeepParse does not distinguish between state, province, region or country
    "Province": "Country",
    "PostalCode" : "PostalCode"
}

class DeepParseParser:
    def __init__(self, **kwargs):
        self.parser = deepparse.parser.AddressParser(device=device, **kwargs)

    def _transform_results(self, parsed_addresses):
        results = []
        for addr in parsed_addresses:
            result_builder = ParsedAddressResultBuilder()
            tuples = addr.to_list_of_tuples()
            for part, label in tuples:
                result_builder.add_part(DEEPPARSE_LABEL_MAPPING.get(label, label), part)
            results.append(result_builder.build())
        return results

    
    def parse_addresses(self, addresses):
        parsed_results = self.parser(addresses)
        return self._transform_results(parsed_results)
    


In [89]:
import http.client
import json
import urllib.parse
import time

LIBPOSTAL_LABEL_MAPPING = {
    "house_number": "HouseNumber",
    "road": "StreetName",
    "city": "City",
    "state": "State",
    "country": "Country",
    "postcode": "PostalCode"
}

class LibpostalClient:
    def __init__(self, url: str = "http://localhost:7272", label_mapping = LIBPOSTAL_LABEL_MAPPING, start_container_if_unavailable: bool = True):
        self.url = url
        parsed_url = urllib.parse.urlparse(url)
        self.host = parsed_url.hostname
        self.port = parsed_url.port
        self.label_mapping = label_mapping or {}
        self.auto_started = False
        if start_container_if_unavailable:
            if not self.start_container_if_needed():
                raise ConnectionError(f"Could not connect to libpostal server at {self.url}, and failed to start docker container.")
    
    def _transform_results(self, parsed_addresses : list[list[list[str]]]) -> list[dict]:
        results = []
        for addr in parsed_addresses:
            result_builder = ParsedAddressResultBuilder()
            for part, label in addr:
                if label in self.label_mapping:
                    label = self.label_mapping[label]
                result_builder.add_part(label, part)
            results.append(result_builder.build())
        return results
    
    def _health_check(self) -> bool:
        conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
        try:
            conn.request("GET", "/health")
            response = conn.getresponse()
            conn.close()
            return response.status == 204
        except Exception:
            return False
    
    def _start_docker_container(self) -> bool:
        print("Attempting to start libpostal-server docker-compose service...")
        print("This may take a long time on first run since the docker image needs to be built.")
        import subprocess
        result = subprocess.run(["docker-compose", "-f", "docker-compose.yml", "up", "-d", "libpostal-server"], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Failed to start libpostal-server docker container (exit code {result.returncode}):")
            print(result.stdout)
            print(result.stderr)
            return False
        for _ in range(10):
                time.sleep(1)
                if self._health_check():
                    print("Libpostal server is now available.")
                    self.auto_started = True
                    return True
        return False
    
    def start_container_if_needed(self):
        if not self._health_check():
            return self._start_docker_container()
        else: return True
            

    def parse_addresses(self, addresses: list[str]):
        def _request(addresses):
            conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
            headers = {'Content-Type': 'application/json'}
            if isinstance(addresses, str):
                addresses = [addresses]
            elif not isinstance(addresses, list):
                addresses = list(addresses)
            body = json.dumps(addresses)
            
            conn.request("GET", "/parse_addresses", body, headers)
            response = conn.getresponse()
            data = response.read()
            
            results = json.loads(data.decode('utf-8'))
            conn.close()

            return self._transform_results(results)
        try:
            return _request(addresses)
        except Exception as e:
            if self._health_check():
                raise e
            else:
                print(f"Libpostal server not reachable at {self.url}.")
                if not self._start_docker_container():
                    raise e
            return _request(addresses)
    
    def close(self):
        if self.auto_started:
            print("Stopping auto-started libpostal-server docker container...")
            import subprocess
            subprocess.run(["docker-compose", "-f", "docker-compose.yml", "down", "libpostal-server"])

In [90]:
import transformers
from transformers import pipeline

class LlamaAddressParsingModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=32):
        tokenizer = transformers.AutoTokenizer.from_pretrained(
            "meta-llama/Llama-3.2-1B-Instruct", padding_side='left', device=device)
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x : x)

    def parse_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt % {"address" : address}}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"]) for r in result]
        return responses

def extract_json_block(model_response : str):
    limit_chars = [('{', '}'), ('[', ']'), ('"', '"')]
    json_str = model_response
    parts = model_response.split("```")
    if len(parts) >= 2: # single code block or malformed code block
        json_str = parts[1]
    elif len(parts) >= 3:
        for part in parts:
            part = part.strip()
            if part.startswith("json") or any(part.startswith(lim[0]) and part.endswith(lim[1]) for lim in limit_chars):
                json_str = part
                break
    if json_str.startswith("json"):
        json_str = json_str[4:].strip()
    return json_str

In [91]:
import json

class JSONObjectParser:
    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            obj = json.loads(json_str)
            result_builder = ParsedAddressResultBuilder()
            for k, v in obj.items():
                result_builder.add_part(k, v)
            result_builder.set_reserved("fullResponse", response)
            data = result_builder.build()
        except Exception as e:
            data = {"fullResponse": response, "error": str(e)}
        return data

class JSONTuplesParser:
    def __init__(self, ignore_other = True):
        self.ignore_other = ignore_other

    def __call__(self, response: str) -> dict:
        ignore_key = None
        if self.ignore_other:
            ignore_key = "Other"
            if isinstance(self.ignore_other, str):
                ignore_key = self.ignore_other
        try:
            json_str = extract_json_block(response)
            tuples = json.loads(json_str)
            result_builder = ParsedAddressResultBuilder()
            for part, ptype in tuples:
                if ptype != ignore_key:
                    result_builder.add_part(ptype, part)
            result_builder.set_reserved("fullResponse", response)
            data = result_builder.build()
        except Exception as e:
            data = {"fullResponse": response, "error": str(e)}
        return data

In [92]:
from collections import OrderedDict
import json
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv")
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv")

EXAMPLES = [
    ("Berlin, Alexanderplatz 1, 10178", 
     OrderedDict([("City" , "Berlin"), ("StreetName", "Alexanderplatz"), ("HouseNumber", "1")])),
    ("Braunschweig, Uferstr. 25", # From BZK open training set
     OrderedDict([("City", "Braunschweig"), ("StreetName", "Uferstr."), ("HouseNumber", "25")])),
    ("808 Westend Avenue, New York 25, N.Y.", # From BZK open training set
        OrderedDict([("StreetName", "Westend Avenue"), ("HouseNumber", "808"), 
        ("City", "New York"), ("State", "N.Y.")])),
]

PROMPTS = [
    "Segment addresses into their components.\n"
    "Output only a JSON object with the following keys: " + ", ".join(COLS_TO_PREDICT) + ". "
    "Do not include fields that cannot be determined and do not try to guess their values. "
    "For example, if the address is simply \"Berlin\" then the field \"Country\" should be null. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps(ex)}\n```\n" for adr, ex in EXAMPLES) +
    "Now segment the following address:\n%(address)s",

    "Annotate address components with the respective types. "
    "Consider the component types: " + ", ".join(COLS_TO_PREDICT + ["Other"]) + ". "
    "Not all addresses will contain all component types and you must not guess the missing ones. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Output only a JSON list of [component, type] tuples. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps([(v,k) for k,v in ex.items()])}\n```\n" for adr, ex in EXAMPLES) +
    "Now annotate the following address:\n%(address)s",
]
for i, prompt in enumerate(PROMPTS):
    print(f"Prompt {i}:")
    print(prompt)
    print()

Prompt 0:
Segment addresses into their components.
Output only a JSON object with the following keys: HouseNumber, StreetName, City, State, Country. Do not include fields that cannot be determined and do not try to guess their values. For example, if the address is simply "Berlin" then the field "Country" should be null. Addresses will most times be written in german, meaning country and city names may be in german and the addresses may include german terms such as:
 - "burg" or "stadt" for city
 - "straße" or its abbreviation "str." for street.
These terms may occur as a suffix to another word. Consider the following examples:
Address: Berlin, Alexanderplatz 1, 10178
```json
{"City": "Berlin", "StreetName": "Alexanderplatz", "HouseNumber": "1"}
```
Address: Braunschweig, Uferstr. 25
```json
{"City": "Braunschweig", "StreetName": "Uferstr.", "HouseNumber": "25"}
```
Address: 808 Westend Avenue, New York 25, N.Y.
```json
{"StreetName": "Westend Avenue", "HouseNumber": "808", "City": "Ne

In [93]:

import time
from pathlib import Path

model_configs = [
    {
        "name" : "libpostal",
        "factory": lambda: LibpostalClient(),
        "cleanup": lambda client: client.close(),
    },
    {
        "name" : "deepparse-bpemb",
        "factory": lambda: DeepParseParser(model_type="bpemb"),
    },
    {
        "name" : "deepparse-fasttext",
        "factory": lambda: DeepParseParser(model_type="fasttext"),
    },
    {
        "name" : "deepparse-bpemb-attention",
        "factory": lambda: DeepParseParser(model_type="bpemb", attention_mechanism=True),
    },
    {
        "name" : "deepparse-fasttext-attention",
        "factory": lambda: DeepParseParser(model_type="fasttext", attention_mechanism=True),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt0",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt0",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt1",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt1",
        "factory": lambda: LlamaAddressParsingModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
]



def eval(dataset, configs, cols_to_predict, pred_cache_path=None):
    if pred_cache_path is not None:
        pred_cache_path = Path(pred_cache_path)
    if pred_cache_path is None or not pred_cache_path.exists():
        cached_preds = {}
    else:
        print(f"Loading cached predictions...")
        with open(pred_cache_path, "r") as f:
            cached_preds = json.load(f)
    
    preds_per_config = []
    model_results = []

    for config in configs:
        config_name = config["name"]
        if config_name in cached_preds:
            print(f"Using cached predictions for model {config_name}... To avoid this delete or rename the file {pred_cache_path} or delete the entry for {config_name} inside it.")
            preds = cached_preds[config_name]["preds"]
            deltatime = cached_preds[config_name]["deltatime"]
        else:
            print(f"Loading model {config_name}...")
            model = config["factory"]()
            print(f"Segmenting addresses...")
            start = time.monotonic()
            preds = model.parse_addresses(dataset["FullAddress"].tolist())
            deltatime = time.monotonic() - start
            if "cleanup" in config:
                print("Cleaning up model resources...")
                config["cleanup"](model)
            print("Parsing model responses...")
            if pred_cache_path is not None:
                cached_preds[config_name] = {
                    "preds": preds,
                    "deltatime": deltatime
                }
        preds_df = pd.DataFrame(preds)
        preds_per_config.append(preds_df)
        print("Computing metrics...")
        metrics = compare_preds(preds_df, dataset[cols_to_predict], target_columns=cols_to_predict)
        metrics["deltatime"] = deltatime
        metrics["rate"] = len(dataset) / metrics["deltatime"]
        metrics["error"] = preds_df["error"].notna().sum() if "error" in preds_df.columns else 0
        metrics["errorRate"] = metrics["error"] / len(dataset)
        preds_df["config_name"] = config_name
        preds_df["FullAddress"] = dataset["FullAddress"]

        model_results.append(metrics)
        print(f"Results for model {config_name}: {metrics}")

    if pred_cache_path is not None:
        with open(pred_cache_path, "w") as f:
            json.dump(cached_preds, f)

    preds_per_config_df = pd.concat(preds_per_config)
    default_cols = ["config_name", "FullAddress"] + cols_to_predict
    new_order = default_cols + [col for col in preds_per_config_df.columns if col not in default_cols]
    preds_per_config_df = preds_per_config_df[new_order]

    results_df = pd.DataFrame(model_results, index = [config["name"] for config in configs])
    return preds_per_config_df, results_df



preds_per_config, model_statistics = eval(bzkopen_val, model_configs, COLS_TO_PREDICT, pred_cache_path=Path("cached_preds_val.json"))

model_statistics

Loading cached predictions...
Using cached predictions for model libpostal... To avoid this delete or rename the file cached_preds_val.json or delete the entry for libpostal inside it.
Computing metrics...
Results for model libpostal: OrderedDict({'accuracy': 0.7681818181818182, 'precision': 0.6568047337278107, 'recall': 0.45121951219512196, 'f1': 0.5349397590361447, 'accuracy_with_tol_1': 0.7742424242424243, 'accuracy_with_tol_2': 0.7803030303030303, 'accuracy_with_tol_3': 0.7863636363636364, 'accuracy_with_tol_4': 0.8136363636363636, 'average_levenshtein': 1.7287878787878788, 'average_similarity': 0.7922009480532585, 'average_levenshtein_match': 2.0974264705882355, 'average_similarity_match': 0.9611261502116738, 'no_match_rate': 0.17575757575757578, 'deltatime': 1.187999999994645, 'rate': 111.11111111161196, 'error': 0, 'errorRate': 0.0})
Using cached predictions for model deepparse-bpemb... To avoid this delete or rename the file cached_preds_val.json or delete the entry for deeppar

,accuracy,precision,recall,f1,accuracy_with_tol_1,accuracy_with_tol_2,accuracy_with_tol_3,accuracy_with_tol_4,average_levenshtein,average_similarity,average_levenshtein_match,average_similarity_match,no_match_rate,deltatime,rate,error,errorRate
libpostal,0.768182,0.656805,0.451220,0.534940,0.774242,0.780303,0.786364,0.813636,1.728788,0.792201,2.097426,0.961126,0.175758,1.188,111.111111,0,0.000000
deepparse-bpemb,0.457576,0.289100,0.247967,0.266958,0.459091,0.471212,0.481818,0.496970,3.281818,0.509956,5.363409,0.843536,0.395455,0.188,702.127660,0,0.000000
deepparse-fasttext,0.501515,0.360000,0.329268,0.343949,0.509091,0.518182,0.528788,0.551515,2.519697,0.555887,3.851765,0.863259,0.356061,1.453,90.846524,0,0.000000
deepparse-bpemb-attention,0.365152,0.084270,0.060976,0.070755,0.375758,0.406061,0.428788,0.460606,3.400000,0.396237,6.974843,0.822378,0.518182,0.906,145.695364,0,0.000000
deepparse-fasttext-attention,0.322727,0.137652,0.138211,0.137931,0.330303,0.350000,0.369697,0.398485,4.089394,0.355572,8.969799,0.787508,0.548485,0.141,936.170213,0,0.000000
Llama-3.2-1B-Instruct-prompt0,0.518182,0.353365,0.597561,0.444109,0.560606,0.578788,0.603030,0.695455,3.096970,0.541067,5.378947,0.939748,0.424242,111.750,1.181208,2,0.015152
Llama-3.2-3B-Instruct-prompt0,0.822727,0.648551,0.727642,0.685824,0.824242,0.833333,0.845455,0.883333,1.215152,0.842473,1.382759,0.958676,0.121212,231.250,0.570811,3,0.022727
Llama-3.2-1B-Instruct-prompt1,0.777273,0.584000,0.593496,0.588710,0.787879,0.795455,0.815152,0.846970,1.663636,0.811336,1.946809,0.949435,0.145455,128.734,1.025370,14,0.106061
Llama-3.2-3B-Instruct-prompt1,0.834848,0.738532,0.654472,0.693966,0.839394,0.842424,0.854545,0.881818,1.187879,0.857669,1.344768,0.970947,0.116667,336.516,0.392255,3,0.022727


In [94]:
preds_per_config

,config_name,FullAddress,HouseNumber,StreetName,City,State,Country,house,state_district,unit,...,HouseNumberPart,error,Direction,Postal,CountryCode,City/Town,State/Region,StreetName/Other,District,PlaceName
0,libpostal,"Regensburg, Königstr. 2/I",2/i,königstr.,NaN,NaN,NaN,regensburg,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,dortmund,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,jöhlingen/krs. durlach/baden.,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,burlington road manchester,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,NaN,leer/ostfriesland,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,NaN,NaN,/Polen,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,St.,Jackson Heights,N.Y.,USA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,NaN,Losone,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [95]:
default_cols = ["config_name", "FullAddress"] + COLS_TO_PREDICT
preds_vs_trues = preds_per_config[default_cols].merge(
    bzkopen_val[default_cols[1:]], on="FullAddress", suffixes=("_pred", "_true"), how="left")
preds_vs_trues = preds_vs_trues[["config_name", "FullAddress"] + [new_col for col in COLS_TO_PREDICT for new_col in [f"{col}_pred", f"{col}_true"]]] # Order the columns for readability
preds_vs_trues

,config_name,FullAddress,HouseNumber_pred,HouseNumber_true,StreetName_pred,StreetName_true,City_pred,City_true,State_pred,State_true,Country_pred,Country_true
0,libpostal,"Regensburg, Königstr. 2/I",2/i,2/I,königstr.,Königstr.,NaN,Regensburg,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,NaN,NaN,dortmund,Dortmund,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,Jöhlingen,NaN,Baden,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,8,burlington road manchester,Burlington Road,NaN,Manchester,NaN,NaN,NaN,England
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,Leer,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1291,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,NaN,NaN,NaN,Sosnowice,NaN,NaN,/Polen,Polen
1292,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,NaN,St.,NaN,Jackson Heights,St. Jackson Heights,N.Y.,N.Y.,USA,USA
1293,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,NaN,NaN,NaN,Losone,Losone,NaN,NaN,NaN,CSR
1294,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rum.


In [102]:
metric = "recall"
metric_per_column = pd.DataFrame(index=model_statistics.index, columns=COLS_TO_PREDICT)
for col in COLS_TO_PREDICT:
    for config in model_statistics.index:
        preds = preds_per_config[preds_per_config["config_name"] == config]
        if col in preds.columns:
            metric_per_column.at[config, col] = compare_preds(preds, bzkopen_val, target_columns=[col])[metric]
        else:
            metric_per_column.at[config, col] = pd.NA
print(f"Per-column {metric} scores:")
metric_per_column

Per-column recall scores:


,HouseNumber,StreetName,City,State,Country
libpostal,0.75,0.65,0.333333,0.5,0.3
deepparse-bpemb,0.625,0.05,0.258333,0.0,0.075
deepparse-fasttext,0.625,0.1,0.375,0.0,0.175
deepparse-bpemb-attention,0.075,0.025,0.008333,0.0,0.25
deepparse-fasttext-attention,0.275,0.025,0.083333,0.0,0.3
Llama-3.2-1B-Instruct-prompt0,0.675,0.625,0.725,0.833333,0.075
Llama-3.2-3B-Instruct-prompt0,0.7,0.825,0.675,0.666667,0.825
Llama-3.2-1B-Instruct-prompt1,0.6,0.525,0.591667,0.666667,0.65
Llama-3.2-3B-Instruct-prompt1,0.75,0.7,0.608333,0.5,0.675


In [103]:
bzk_fields = bzkopen_val["field"].unique()
print(f"bzk_fields: {bzk_fields}")
metric_per_bzk_field = pd.DataFrame(index=model_statistics.index, columns=bzk_fields)

for field in bzk_fields:
    mask = bzkopen_val["field"] == field
    subset = bzkopen_val[mask]
    for config in model_statistics.index:
        preds = preds_per_config[preds_per_config["config_name"] == config]
        metric_per_bzk_field.at[config, field] = compare_preds(preds[mask], subset, target_columns=COLS_TO_PREDICT)[metric]
print(f"Per-BZK-field {metric} scores:")
metric_per_bzk_field

bzk_fields: <StringArray>
['ApplicantCurrentAddress',        'VictimBirthPlace',
     'ApplicantBirthPlace',        'VictimDeathPlace']
Length: 4, dtype: str
Per-BZK-field recall scores:


,ApplicantCurrentAddress,VictimBirthPlace,ApplicantBirthPlace,VictimDeathPlace
libpostal,0.455782,0.416667,0.450704,0.5
deepparse-bpemb,0.190476,0.416667,0.295775,0.5
deepparse-fasttext,0.244898,0.666667,0.380282,0.5
deepparse-bpemb-attention,0.061224,0.041667,0.070423,0.0
deepparse-fasttext-attention,0.136054,0.208333,0.126761,0.0
Llama-3.2-1B-Instruct-prompt0,0.612245,0.666667,0.521127,1.0
Llama-3.2-3B-Instruct-prompt0,0.755102,0.708333,0.676056,0.75
Llama-3.2-1B-Instruct-prompt1,0.632653,0.583333,0.521127,0.5
Llama-3.2-3B-Instruct-prompt1,0.653061,0.708333,0.633803,0.75
